# $c$ ed $s$ meglio ricavarli da fit segnale originale

---

Leggo

In [1]:
import h5py
import numpy as np
from scipy.optimize import curve_fit
from scipy.signal import find_peaks
import time

# Percorso del file .mat
file_path = "../data/50k/odmr_easyspin_raw_50000.mat"

# -------------------------------------------------------------------------
# 1. LETTURA DEGLI SPETTRI E ASSE FREQUENZE
# -------------------------------------------------------------------------
with h5py.File(file_path, "r") as f:
    # MATLAB memorizza per colonne: trasponiamo per avere [N_spettri, N_punti]
    spectra_data = np.array(f["spectra_data"]).T
    mw_freqs = np.array(f["mw_freqs"]).squeeze()

N_spectra = spectra_data.shape[0]
f_ghz = np.asarray(mw_freqs, dtype=np.float64)
f_min, f_max = float(f_ghz.min()), float(f_ghz.max())
delta_f = f_max - f_min

# -------------------------------------------------------------------------
# 2. DEFINIZIONE DEL FIT SUL SEGNALE PULITO (DUE PICCHI POSITIVI)
# -------------------------------------------------------------------------
def double_lorentzian_absorption(f, y0, A1, x1, w1, A2, x2, w2):
    """Modello a due lorentziane positive per assorbimento puro EasySpin."""
    return (
        y0
        + A1 * (w1**2 / ((f - x1)**2 + w1**2))
        + A2 * (w2**2 / ((f - x2)**2 + w2**2))
    )

def estrai_dip_singolo(f, y_raw):
    """
    Ricava le posizioni fisiche x1 e x2 (in GHz) dal profilo puro EasySpin.
    Ritorna: c_norm, s_norm, x1_ghz, x2_ghz
    """
    # 1. Ricerca dei picchi iniziali
    peaks, _ = find_peaks(y_raw, height=0.15, distance=6)
    
    if len(peaks) >= 2:
        top_two = peaks[np.argsort(y_raw[peaks])[-2:]]
        x1_guess, x2_guess = np.sort(f[top_two])
    else:
        # Fallback nel caso in cui i picchi siano molto vicini
        mid = len(f) // 2
        x1_guess = f[np.argmax(y_raw[:mid])]
        x2_guess = f[mid + np.argmax(y_raw[mid:])]
        
    p0 = [
        0.0,
        float(y_raw.max()), x1_guess, 0.005,
        float(y_raw.max()), x2_guess, 0.005
    ]
    
    bounds = (
        [-0.05, 0.0, f_min, 0.001, 0.0, f_min, 0.001],
        [ 0.15, 2.0, f_max, 0.020, 2.0, f_max, 0.020]
    )
    
    try:
        popt, _ = curve_fit(
            double_lorentzian_absorption,
            f,
            y_raw,
            p0=p0,
            bounds=bounds,
            maxfev=1500
        )
        x1, x2 = sorted([popt[2], popt[5]])
    except Exception:
        x1, x2 = sorted([x1_guess, x2_guess])
        
    # Coordinate normalizzate nello spazio adimensionale [0, 1] della rete
    x1_norm = (x1 - f_min) / delta_f
    x2_norm = (x2 - f_min) / delta_f
    
    c_norm = (x1_norm + x2_norm) / 2.0
    s_norm = x2_norm - x1_norm
    
    return c_norm, s_norm, x1, x2

Fitto il segnale con estrai_dip_reali e sovrascrivo le labels nel file .mat

In [2]:
# -------------------------------------------------------------------------
# 3. ESTRAZIONE PER TUTTI GLI SPETTRI E SOVRASCRITTURA SUL FILE .MAT
# -------------------------------------------------------------------------
def estrai_e_sovrascrivi_labels(file_path):
    print(f"Avvio estrazione ground-truth per {N_spectra} spettri puliti...")
    t0 = time.time()
    
    c_veri = np.empty(N_spectra, dtype=np.float64)
    s_veri = np.empty(N_spectra, dtype=np.float64)
    
    for i in range(N_spectra):
        c_norm, s_norm, _, _ = estrai_dip_singolo(f_ghz, spectra_data[i])
        c_veri[i] = c_norm
        s_veri[i] = s_norm
        
        if (i + 1) % 2000 == 0 or i == N_spectra - 1:
            print(f"Elaborati {i + 1}/{N_spectra} spettri (tempo trascorso: {time.time() - t0:.1f}s)")
            
    print(f"\nEstrazione completata in {time.time() - t0:.1f}s.")
    print(f"Range nuovi 'c': [{c_veri.min():.4f}, {c_veri.max():.4f}]")
    print(f"Range nuovi 's': [{s_veri.min():.4f}, {s_veri.max():.4f}]")
    
    # Sovrascrittura diretta nel file HDF5/.mat
    with h5py.File(file_path, "r+") as f:
        # Recuperiamo la forma originaria del dataset salvato da MATLAB (es. [N, 1] o [1, N])
        shape_c = f["labels_c"].shape
        shape_s = f["labels_s"].shape
        
        f["labels_c"][:] = c_veri.reshape(shape_c)
        f["labels_s"][:] = s_veri.reshape(shape_s)
        
    print(f"File '{file_path}' aggiornato con successo.")

# Esegui l'aggiornamento
estrai_e_sovrascrivi_labels(file_path)

Avvio estrazione ground-truth per 50000 spettri puliti...
Elaborati 2000/50000 spettri (tempo trascorso: 4.5s)
Elaborati 4000/50000 spettri (tempo trascorso: 8.3s)
Elaborati 6000/50000 spettri (tempo trascorso: 12.4s)
Elaborati 8000/50000 spettri (tempo trascorso: 16.6s)
Elaborati 10000/50000 spettri (tempo trascorso: 20.4s)
Elaborati 12000/50000 spettri (tempo trascorso: 24.6s)
Elaborati 14000/50000 spettri (tempo trascorso: 29.1s)
Elaborati 16000/50000 spettri (tempo trascorso: 32.9s)
Elaborati 18000/50000 spettri (tempo trascorso: 37.2s)
Elaborati 20000/50000 spettri (tempo trascorso: 40.9s)
Elaborati 22000/50000 spettri (tempo trascorso: 44.5s)
Elaborati 24000/50000 spettri (tempo trascorso: 48.1s)
Elaborati 26000/50000 spettri (tempo trascorso: 51.6s)
Elaborati 28000/50000 spettri (tempo trascorso: 55.2s)
Elaborati 30000/50000 spettri (tempo trascorso: 58.6s)
Elaborati 32000/50000 spettri (tempo trascorso: 62.1s)
Elaborati 34000/50000 spettri (tempo trascorso: 66.6s)
Elaborati 360